# Polymarket: Analysis of Proposed and Disputed Market Resolutions

This notebook provides a real-time list of Polymarket markets that have an **active proposed resolution** but have not yet been permanently settled or resolved. 

### Key Features
1. **Dynamic Filtering**: Fetches only active, liquid markets (defined as >$250 24h volume) to ensure findings are tradeable.
2. **Implied Proposal Logic**: Since the public REST API does not explicitly return the proposed outcome string ("YES" or "NO"), this notebook uses a **price-based heuristic** to infer the proposal.
3. **Premium Calculation**: Calculates the potential profit (premium) based on the distance between the current price and the implied payout (0 or 100 cents).

---

In [41]:
import ast
import math
import requests
import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

In [42]:
GAMMA_MARKETS_URL = 'https://gamma-api.polymarket.com/markets'
PAGE_LIMIT = 100
MIN_24H_VOLUME = 250.0  # Minimum 24-hour volume threshold for tradability


def to_float(value):
    try:
        if value is None or value == '':
            return math.nan
        return float(value)
    except (TypeError, ValueError):
        return math.nan


def parse_statuses(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else [str(parsed)]
        except (ValueError, SyntaxError):
            return [value]
    return [str(value)]


def fetch_tradable_markets(min_volume=MIN_24H_VOLUME, page_limit=PAGE_LIMIT):
    """
    Fetches active, unclosed markets from the API sorted by 24h volume (highest first).
    Stops paginating as soon as the API returns markets below our minimum volume threshold.
    """
    tradable_markets = []
    session = requests.Session()
    offset = 0

    while True:
        params = {
            'limit': page_limit,
            'offset': offset,
            'active': 'true',
            'closed': 'false',
            'order': 'volume24hr',
            'ascending': 'false'
        }
        response = session.get(GAMMA_MARKETS_URL, params=params, timeout=30)
        response.raise_for_status()
        batch = response.json()
        
        if not batch:
            break
            
        for market in batch:
            vol = to_float(market.get('volume24hr'))
            if pd.isna(vol) or vol < min_volume:
                return tradable_markets
            
            tradable_markets.append(market)

        if len(batch) < page_limit:
            break
            
        offset += page_limit

    return tradable_markets


def build_proposed_markets_table(markets):
    records = []

    for market in markets:
        statuses = parse_statuses(market.get('umaResolutionStatuses'))
        statuses_lower = [str(status).lower() for status in statuses]

        # Look for markets that have a proposed resolution
        if 'proposed' not in statuses_lower:
            continue

        # Exclude markets that appear already settled or resolved
        if any(status in statuses_lower for status in ['resolved', 'settled']):
            continue

        last_price = to_float(market.get('lastTradePrice'))
        best_bid = to_float(market.get('bestBid'))
        best_ask = to_float(market.get('bestAsk'))
        spread = to_float(market.get('spread'))
        volume_24h = to_float(market.get('volume24hr'))

        # Require both sides of the market to be present
        if pd.isna(best_bid) or pd.isna(best_ask):
            continue

        # If we can't determine the price, skip
        if pd.isna(last_price):
            continue

        # Infer the proposed resolution from the market price
        implied_proposed_yes = last_price >= 0.50
        implied_resolution_str = "Yes" if implied_proposed_yes else "No"
        
        # Calculate premium based on implied proposed resolution
        if implied_proposed_yes:
            premium_cents = (1.0 - last_price) * 100
        else:
            premium_cents = last_price * 100

        slug = market.get('slug')

        record = {
            'market_id': market.get('id'),
            'question': market.get('question'),
            'slug': slug,
            'url': f"https://polymarket.com/market/{slug}" if slug else None,
            'uma_resolution_statuses': ', '.join(statuses),
            'implied_proposal': implied_resolution_str,
            'active': market.get('active'),
            'closed': market.get('closed'),
            'last_price': last_price,
            'last_price_cents': last_price * 100,
            'premium_cents': premium_cents,
            'best_bid': best_bid,
            'best_ask': best_ask,
            'bid_ask_spread_cents': spread * 100 if pd.notna(spread) else math.nan,
            'volume_24h': volume_24h,
        }
        records.append(record)

    df = pd.DataFrame(records)
    if df.empty:
        return df

    # Sort descending by premium so the greatest premium is at the top
    df = df.sort_values(['premium_cents', 'volume_24h'], ascending=[False, False], na_position='last').reset_index(drop=True)
    return df

## Data Acquisition and Processing

We fetch active, unclosed markets from the Gamma API, sorted by 24h volume. To optimize performance, we stop fetching once the volume falls below our $250 threshold. 

We then filter for markets where:
- UMA status is `proposed`.
- UMA status is **not** `resolved` or `settled`.
- Both Bid and Ask prices exist.

The **Implied Proposal** and **Premium** are calculated as follows:
- **If Price $\ge$ 50¢**: We infer the proposal is **YES**. 
    - *Premium = 100 - Price*
- **If Price < 50¢**: We infer the proposal is **NO**. 
    - *Premium = Price - 0* (or simply the current price).

In [43]:
# Fetch only tradable, active markets sorted by volume until volume drops below $250
liquid_markets = fetch_tradable_markets(min_volume=250.0)
proposed_df = build_proposed_markets_table(liquid_markets)

print(f'Total liquid markets fetched (> $250 24h volume): {len(liquid_markets):,}')
print(f'Active proposed markets (not settled/resolved): {len(proposed_df):,}')

proposed_df

Total liquid markets fetched (> $250 24h volume): 6,031
Active proposed markets (not settled/resolved): 153


,market_id,question,slug,url,uma_resolution_statuses,implied_proposal,active,closed,last_price,last_price_cents,premium_cents,best_bid,best_ask,bid_ask_spread_cents,volume_24h
0,2092759,Counter-Strike: Rare Atom vs Legion - Map 1 Winner,cs2-ra1-legion-2026-04-27-game1,https://polymarket.com/market/cs2-ra1-legion-2026-04-27-game1,proposed,Yes,True,False,0.500,50.0,50.0,0.500,0.510,1.0,2.302184e+03
1,2092668,Counter-Strike: Walczaki vs CYBERSHOKE Prospects - Map 1 Winner,cs2-wal2-cyb-2026-04-27-game1,https://polymarket.com/market/cs2-wal2-cyb-2026-04-27-game1,proposed,Yes,True,False,0.500,50.0,50.0,0.500,0.510,1.0,9.671042e+02
2,2092739,Counter-Strike: Just Swing vs Chinggis Warriors - Map 1 Winner,cs2-js-cw-2026-04-27-game1,https://polymarket.com/market/cs2-js-cw-2026-04-27-game1,proposed,Yes,True,False,0.510,51.0,49.0,0.490,0.510,2.0,9.548705e+02
3,2037907,Clavicular pregnancy in 2026?,clavicular-pregnancy-in-2026,https://polymarket.com/market/clavicular-pregnancy-in-2026,"proposed, disputed, proposed, disputed",Yes,True,False,0.730,73.0,27.0,0.730,0.750,2.0,3.024566e+05
4,1910397,"Will Iran strike Bahrain by April 30, 2026?",will-iran-strike-bahrain-by-april-30-2026-553,https://polymarket.com/market/will-iran-strike-bahrain-by-april-30-2026-553,"proposed, disputed, proposed, disputed",No,True,False,0.120,12.0,12.0,0.120,0.130,1.0,3.463269e+04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,2034582,Will the price of XRP be above $1.20 on April 27?,xrp-above-1pt2-on-april-27,https://polymarket.com/market/xrp-above-1pt2-on-april-27,proposed,Yes,True,False,0.999,99.9,0.1,0.999,1.000,0.1,2.505840e+02
149,2036399,"US x Iran ceasefire extended by April 22, 2026?",us-x-iran-ceasefire-extended-by-april-22-2026,https://polymarket.com/market/us-x-iran-ceasefire-extended-by-april-22-2026,"proposed, disputed, proposed, disputed, proposed, disputed",No,True,False,0.001,0.1,0.1,0.001,0.002,0.1,2.446915e+07
150,2036789,Will the next model released by OpenAI debut at a score of at least 1520?,will-the-next-model-released-by-openai-debut-at-a-score-of-at-least-1520-545,https://polymarket.com/market/will-the-next-model-released-by-openai-debut-at-a-score-of-at-least-1520-545,proposed,No,True,False,0.001,0.1,0.1,0.001,0.004,0.3,1.009953e+04
151,2036789,Will the next model released by OpenAI debut at a score of at least 1520?,will-the-next-model-released-by-openai-debut-at-a-score-of-at-least-1520-545,https://polymarket.com/market/will-the-next-model-released-by-openai-debut-at-a-score-of-at-least-1520-545,proposed,No,True,False,0.001,0.1,0.1,0.001,0.006,0.5,9.028439e+03


## Market Opportunity Analysis

The table below lists the proposed markets sorted by **Premium Cents** (descending). 

### How to read the results:
- **Implied Proposal**: The assumed resolution based on the current price. 
- **Premium Cents**: The estimated profit per share if the proposal is finalized. 
    - *Note: Higher premiums often indicate that the market expects a dispute or that the proposal is controversial.*
- **Bid/Ask Spread**: Useful for checking entry/exit costs. A spread close to the premium may make the trade unprofitable.
- **24h Volume**: Indicates liquidity. Since we filtered for $>\$250$, these markets have at least some recent activity.

In [44]:
if not proposed_df.empty:
    summary_columns = [
        'question',
        'implied_proposal',
        'last_price_cents',
        'premium_cents',
        'best_bid',
        'best_ask',
        'bid_ask_spread_cents',
        'volume_24h',
        'uma_resolution_statuses',
        'url',
    ]

    display_df = proposed_df[summary_columns].copy()
    numeric_cols = ['last_price_cents', 'premium_cents', 'best_bid', 'best_ask', 'bid_ask_spread_cents', 'volume_24h']
    display_df[numeric_cols] = display_df[numeric_cols].round(2)
    display(display_df)
else:
    print('No proposed markets were returned by the API at this time.')

,question,implied_proposal,last_price_cents,premium_cents,best_bid,best_ask,bid_ask_spread_cents,volume_24h,uma_resolution_statuses,url
0,Counter-Strike: Rare Atom vs Legion - Map 1 Winner,Yes,50.0,50.0,0.50,0.51,1.0,2302.18,proposed,https://polymarket.com/market/cs2-ra1-legion-2026-04-27-game1
1,Counter-Strike: Walczaki vs CYBERSHOKE Prospects - Map 1 Winner,Yes,50.0,50.0,0.50,0.51,1.0,967.10,proposed,https://polymarket.com/market/cs2-wal2-cyb-2026-04-27-game1
2,Counter-Strike: Just Swing vs Chinggis Warriors - Map 1 Winner,Yes,51.0,49.0,0.49,0.51,2.0,954.87,proposed,https://polymarket.com/market/cs2-js-cw-2026-04-27-game1
3,Clavicular pregnancy in 2026?,Yes,73.0,27.0,0.73,0.75,2.0,302456.57,"proposed, disputed, proposed, disputed",https://polymarket.com/market/clavicular-pregnancy-in-2026
4,"Will Iran strike Bahrain by April 30, 2026?",No,12.0,12.0,0.12,0.13,1.0,34632.69,"proposed, disputed, proposed, disputed",https://polymarket.com/market/will-iran-strike-bahrain-by-april-30-2026-553
...,...,...,...,...,...,...,...,...,...,...
148,Will the price of XRP be above $1.20 on April 27?,Yes,99.9,0.1,1.00,1.00,0.1,250.58,proposed,https://polymarket.com/market/xrp-above-1pt2-on-april-27
149,"US x Iran ceasefire extended by April 22, 2026?",No,0.1,0.1,0.00,0.00,0.1,24469148.50,"proposed, disputed, proposed, disputed, proposed, disputed",https://polymarket.com/market/us-x-iran-ceasefire-extended-by-april-22-2026
150,Will the next model released by OpenAI debut at a score of at least 1520?,No,0.1,0.1,0.00,0.00,0.3,10099.53,proposed,https://polymarket.com/market/will-the-next-model-released-by-openai-debut-at-a-score-of-at-least-1520-545
151,Will the next model released by OpenAI debut at a score of at least 1520?,No,0.1,0.1,0.00,0.01,0.5,9028.44,proposed,https://polymarket.com/market/will-the-next-model-released-by-openai-debut-at-a-score-of-at-least-1520-545


In [45]:
if not proposed_df.empty:
    biggest_premium = proposed_df.iloc[0]
    print('Market with the greatest premium to its implied proposal:')
    print(biggest_premium[['question', 'implied_proposal', 'last_price_cents', 'premium_cents', 'volume_24h', 'best_bid', 'best_ask', 'url']].to_string())

    print('\nTop 5 by premium cents:')
    display(
        proposed_df[
            ['question', 'implied_proposal', 'last_price_cents', 'premium_cents', 'bid_ask_spread_cents', 'volume_24h', 'url']
        ].head(5).round({'last_price_cents': 2, 'premium_cents': 2, 'bid_ask_spread_cents': 2, 'volume_24h': 2})
    )

Market with the greatest premium to its implied proposal:
question                       Counter-Strike: Rare Atom vs Legion - Map 1 Winner
implied_proposal                                                              Yes
last_price_cents                                                             50.0
premium_cents                                                                50.0
volume_24h                                                            2302.184494
best_bid                                                                      0.5
best_ask                                                                     0.51
url                 https://polymarket.com/market/cs2-ra1-legion-2026-04-27-game1

Top 5 by premium cents:


,question,implied_proposal,last_price_cents,premium_cents,bid_ask_spread_cents,volume_24h,url
0,Counter-Strike: Rare Atom vs Legion - Map 1 Winner,Yes,50.0,50.0,1.0,2302.18,https://polymarket.com/market/cs2-ra1-legion-2026-04-27-game1
1,Counter-Strike: Walczaki vs CYBERSHOKE Prospects - Map 1 Winner,Yes,50.0,50.0,1.0,967.10,https://polymarket.com/market/cs2-wal2-cyb-2026-04-27-game1
2,Counter-Strike: Just Swing vs Chinggis Warriors - Map 1 Winner,Yes,51.0,49.0,2.0,954.87,https://polymarket.com/market/cs2-js-cw-2026-04-27-game1
3,Clavicular pregnancy in 2026?,Yes,73.0,27.0,2.0,302456.57,https://polymarket.com/market/clavicular-pregnancy-in-2026
4,"Will Iran strike Bahrain by April 30, 2026?",No,12.0,12.0,1.0,34632.69,https://polymarket.com/market/will-iran-strike-bahrain-by-april-30-2026-553
